# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [10]:

from pathlib import Path

import duckdb
import polars as pl

# Lokasi project
PROJECT_ROOT = Path.cwd().resolve().parent

# Lokasi dataset yang benar
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "SFinDSet.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}"
    )

print(f"Dataset: {DATA_PATH.name}")
print(f"Lokasi: {DATA_PATH}")
print(f"Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB")

# Tes membaca dataset
df = pl.read_csv(DATA_PATH, n_rows=5)

print("\nDataset berhasil dibaca!")
print(df)


Dataset: SFinDSet.csv
Lokasi: /home/jovyan/work/data/raw/SFinDSet.csv
Ukuran file: 112.8 MB

Dataset berhasil dibaca!
shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Transacti ┆ Account_N ┆ Account_N ┆ Transacti ┆ … ┆ Transacti ┆ Transacti ┆ Transacti ┆ Transact │
│ on_ID     ┆ umber     ┆ ame       ┆ on_Amount ┆   ┆ on_Type   ┆ on_Date   ┆ on_Source ┆ ion_Dest │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ination  │
│ str       ┆ i64       ┆ str       ┆ f64       ┆   ┆ str       ┆ str       ┆ i64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ TRANS-588 ┆ 936832620 ┆ David     ┆ 1834602.8 ┆ … ┆ NGN to    ┆ 10/28/202 ┆ 169816110 ┆ 20322866 │
│ 469       ┆ 6         ┆ Howard    ┆           ┆   ┆ USD   

## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [11]:

# Baca dataset CSV secara lazy
lazy_data = pl.scan_csv(DATA_PATH)

# Tampilkan struktur kolom dan tipe datanya
print(lazy_data.collect_schema())

# Hitung jumlah baris dan nilai null setiap kolom
ringkasan = lazy_data.select(
    [
        pl.len().alias("jumlah_baris"),
        *[
            pl.col(kolom).null_count().alias(f"{kolom}_null")
            for kolom in lazy_data.collect_schema().names()
        ],
    ]
).collect()

ringkasan

# Baca dataset CSV secara lazy
lazy_data = pl.scan_csv(DATA_PATH)

# Tampilkan struktur kolom dan tipe datanya
print(lazy_data.collect_schema())

# Hitung jumlah baris dan nilai null setiap kolom
ringkasan = lazy_data.select(
    [
        pl.len().alias("jumlah_baris"),
        *[
            pl.col(kolom).null_count().alias(f"{kolom}_null")
            for kolom in lazy_data.collect_schema().names()
        ],
    ]
).collect()

ringkasan


Schema({'Transaction_ID': String, 'Account_Number': Int64, 'Account_Name': String, 'Transaction_Amount': Float64, 'Transaction_Mode': String, 'Transaction_Type': String, 'Transaction_Date': String, 'Transaction_Source': Int64, 'Transaction_Destination': Int64})
Schema({'Transaction_ID': String, 'Account_Number': Int64, 'Account_Name': String, 'Transaction_Amount': Float64, 'Transaction_Mode': String, 'Transaction_Type': String, 'Transaction_Date': String, 'Transaction_Source': Int64, 'Transaction_Destination': Int64})


jumlah_baris,Transaction_ID_null,Account_Number_null,Account_Name_null,Transaction_Amount_null,Transaction_Mode_null,Transaction_Type_null,Transaction_Date_null,Transaction_Source_null,Transaction_Destination_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1048575,0,0,0,0,0,0,0,0,0


In [12]:
# Ambil 10 baris pertama dari dataset
contoh_data = lazy_data.head(10).collect()

# Tampilkan hasilnya
contoh_data

Transaction_ID,Account_Number,Account_Name,Transaction_Amount,Transaction_Mode,Transaction_Type,Transaction_Date,Transaction_Source,Transaction_Destination
str,i64,str,f64,str,str,str,i64,i64
"""TRANS-588469""",9368326206,"""David Howard""",1834602.8,"""Card Out""","""NGN to USD""","""10/28/2021 2:31""",1698161104,20322866
"""TRANS-239018""",6826045758,"""David Hughes""",537724.4,"""Card Out""","""NGN to USD""","""8/31/2022 23:38""",8172779956,20322866
"""TRANS-1949426""",5386299916,"""David Huber""",7.875963e6,"""Card Out""","""NGN to USD""","""8/20/2023 8:56""",3138972375,20322866
"""TRANS-1588193""",1443563124,"""David Hunt""",2.0079e6,"""Card Out""","""NGN to USD""","""2/13/2023 16:14""",9694080118,20322866
"""TRANS-224390""",7244060183,"""David Hughes""",4.7660e7,"""Card Out""","""NGN to USD""","""3/21/2021 17:10""",6225683619,20322866
"""TRANS-1239929""",9633854457,"""Howard Johnson""",1.1503e6,"""Card Out""","""NGN to USD""","""6/3/2023 19:02""",7868253594,20322866
"""TRANS-1947599""",386730764,"""Holly Williams""",1139611.4,"""Card Out""","""NGN to USD""","""3/19/2022 20:28""",4144644182,20322866
"""TRANS-681022""",4839114133,"""Holly Williams""",2735754.8,"""Card Out""","""NGN to USD""","""11/24/2021 23:57""",1982566656,20322866
"""TRANS-1982400""",558229106,"""Holly Torres""",3814374.1,"""Card Out""","""NGN to USD""","""8/6/2022 20:35""",4413396848,20322866


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [16]:

connection = duckdb.connect()

profil_duckdb = pl.from_pandas(
    connection.execute(
        "SUMMARIZE SELECT * FROM read_csv_auto(?)",
        [str(DATA_PATH)]
    ).fetchdf()
)

profil_duckdb


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,f64
"""Transaction_ID""","""VARCHAR""","""TRANS-1""","""TRANS-999999""",1013351,null,null,null,null,null,1048575,0.0
"""Account_Number""","""BIGINT""","""500""","""9999995264""",1020194,"""4998590299.811408""","""2886773064.090955""","""2500839002""","""4997754223""","""7495323011""",1048575,0.0
"""Account_Name""","""VARCHAR""","""Aaron Abbott""","""Zoe Zavala""",333013,null,null,null,null,null,1048575,0.0
"""Transaction_Amount""","""DOUBLE""","""100.5""","""49999946.5""",1100482,"""13764588.539523637""","""15221204.816237047""","""2276998.897638419""","""4551908.674472427""","""25055333.532327898""",1048575,0.0
"""Transaction_Mode""","""VARCHAR""","""Card Out""","""Transfer Out""",5,null,null,null,null,null,1048575,0.0
"""Transaction_Type""","""VARCHAR""","""NGN to NGN""","""USD to USD""",4,null,null,null,null,null,1048575,0.0
"""Transaction_Date""","""VARCHAR""","""1/1/2020 0:01""","""9/9/2023 9:58""",813717,null,null,null,null,null,1048575,0.0
"""Transaction_Source""","""BIGINT""","""12599""","""9999999517""",1041513,"""4998512684.282938""","""2886117872.4148703""","""2490830092""","""4994090970""","""7494911864""",1048575,0.0
"""Transaction_Destination""","""BIGINT""","""4959""","""9999997549""",1005822,"""4998704461.818359""","""2885133086.3944583""","""2503295695""","""4997898174""","""7495734443""",1048575,0.0


## Catatan Milestone 1 

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

In [17]:

# Temuan awal dataset SFinDSet

# Ukuran file
ukuran_mb = DATA_PATH.stat().st_size / 1_000_000

# Jumlah baris dan tipe kolom
schema = lazy_data.collect_schema()
jumlah_baris = lazy_data.select(pl.len()).collect().item()
tipe_kolom = schema.dtypes()
nama_kolom = schema.names()

# Jumlah null setiap kolom
jumlah_null = lazy_data.select([
    pl.col(kolom).null_count().alias(kolom)
    for kolom in nama_kolom
]).collect()

null_dict = jumlah_null.row(0, named=True)
null_terbanyak = max(null_dict, key=null_dict.get)

print("=== TEMUAN AWAL DATASET SFinDSet ===")
print(f"Ukuran file       : {ukuran_mb:.2f} MB")
print(f"Jumlah baris      : {jumlah_baris:,}")
print(f"Jumlah kolom      : {len(nama_kolom)}")

print("\nTipe kolom:")
for nama, tipe in schema.items():
    print(f"- {nama}: {tipe}")

print("\nKolom dengan null terbanyak:")
print(f"- {null_terbanyak}: {null_dict[null_terbanyak]:,} null")

print("\nJumlah null per kolom:")
for kolom, jumlah in null_dict.items():
    print(f"- {kolom}: {jumlah:,}")

print("\nAnomali awal yang perlu diperiksa:")
print("- Nilai kosong (null) pada setiap kolom.")
print("- Duplikasi baris.")
print("- Ketidaksesuaian tipe data.")
print("- Nilai tidak valid atau di luar rentang yang wajar.")
print("- Format kategori dan teks yang tidak konsisten.")


=== TEMUAN AWAL DATASET SFinDSet ===
Ukuran file       : 112.80 MB
Jumlah baris      : 1,048,575
Jumlah kolom      : 9

Tipe kolom:
- Transaction_ID: String
- Account_Number: Int64
- Account_Name: String
- Transaction_Amount: Float64
- Transaction_Mode: String
- Transaction_Type: String
- Transaction_Date: String
- Transaction_Source: Int64
- Transaction_Destination: Int64

Kolom dengan null terbanyak:
- Transaction_ID: 0 null

Jumlah null per kolom:
- Transaction_ID: 0
- Account_Number: 0
- Account_Name: 0
- Transaction_Amount: 0
- Transaction_Mode: 0
- Transaction_Type: 0
- Transaction_Date: 0
- Transaction_Source: 0
- Transaction_Destination: 0

Anomali awal yang perlu diperiksa:
- Nilai kosong (null) pada setiap kolom.
- Duplikasi baris.
- Ketidaksesuaian tipe data.
- Nilai tidak valid atau di luar rentang yang wajar.
- Format kategori dan teks yang tidak konsisten.
